In [1]:
import os
import openai
from langchain.agents import AgentType
from langchain.chat_models import ChatOpenAI

c:\Users\mbial\AppData\Local\pypoetry\Cache\virtualenvs\gen-ai-common-use-HcUDhRlE-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\mbial\AppData\Local\pypoetry\Cache\virtualenvs\gen-ai-common-use-HcUDhRlE-py3.11\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
llm =  ChatOpenAI(
    model=  "gpt-4.1-nano",
    temperature=0.0,
    max_tokens=256,
    api_key=os.getenv("OPENAI_API_KEY")
)

C:\Users\mbial\AppData\Local\Temp\ipykernel_24472\3831855461.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm =  ChatOpenAI(


## La fonction

Les fonctions sont appelées par les agents pour résoudre des tâches spécifiques. Ces fonction en sont pas des simples fonctions qui seront vues par des développeurs mais des outils qui seront utilisés par une IA. Une bonne documentation est cruciale. Votre fonction doit cocher ces 4 points :
1. **Le but doit être clair** Assurez vous que 

In [11]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    
    result = sum(numbers)
    return {"result": result}

## Tool

In [12]:
from langchain.agents import Tool

In [13]:
add_tool =  Tool(
    name = "AddTool",
    func=add_numbers,
    description="Adds a list of numbers and returns the result."
)

In [14]:
# Tool name
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)


Tool Name:
AddTool
Tool Description:
Adds a list of numbers and returns the result.
Tool Function:
<bound method BaseTool.invoke of Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=<function add_numbers at 0x000002ADBACBF920>)>


In [8]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(add_tool.invoke(test_input)) 

Calling Tool Function:
{'result': 60}


## L'opérateur @tool 

In [15]:
from langchain_core.tools import tool
import re

In [16]:
@tool
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input string.
    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.
    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.
    Example Input:
    "Add the numbers 10, 20, and 30."
    Example Output:
    {"result": 60}
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    
    result = sum(numbers)
    return {"result": result}

In [11]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))

{'result': 60}


## @tool-StructuredTool

In [12]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

Tool Constructor Approach:
Has Schema: True


@tool Decorator Approach:
Has Schema: True
Args Schema Info: {'inputs': {'title': 'Inputs', 'type': 'string'}}


In [17]:
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

In [18]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

In [19]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """
    Adds a list of numbers provided in the input string.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        
    Returns:
        The sum of all numbers found in the input.
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    result = sum(numbers)
    return result

## Initialize agent

In [16]:
from langchain.agents import initialize_agent

In [17]:
agent = initialize_agent(
    tools=[add_tool],
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True,
    handle_parsing_errors=True
)

C:\Users\mbial\AppData\Local\Temp\ipykernel_25304\2980620849.py:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [18]:
Prompt = """
            En 2023, le PIB du Cameroun était de 12 milliards de FCFA. EN 2024, le Cameroun a eu un PIB de 30 milliars de FCFA. 
            Quel est le PIB total entre 2024 et 2023 ?
        """

In [19]:
response =agent.run(Prompt)

C:\Users\mbial\AppData\Local\Temp\ipykernel_25304\205207389.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response =agent.run(Prompt)




> Entering new AgentExecutor chain...
Thought: I need to find the total PIB for both years by adding the PIB of 2023 and 2024. The PIB for 2023 is 12 billion FCFA, and for 2024 it is 30 billion FCFA. I'll add these two numbers to get the total.

Action: AddTool

Action Input: 12 + 30

Observation: {'result': 42}
Thought:Thought: I now know the final answer is the sum of the PIBs for 2023 and 2024, which is 42 billion FCFA.
Observation: Invalid Format: Missing 'Action:' after 'Thought:
Thought:Question: 
            En 2023, le PIB du Cameroun était de 12 milliards de FCFA. EN 2024, le Cameroun a eu un PIB de 30 milliars de FCFA. 
            Quel est le PIB total entre 2024 et 2023 ?
        
Thought: I need to find the total PIB for both years by adding the PIB of 2023 and 2024. The PIB for 2023 is 12 billion FCFA, and for 2024 it is 30 billion FCFA. I'll add these two numbers to get the total.

Action: AddTool

Action Input: 12 + 30

Observation: {'result': 42}
Thought:Final Answer

### **Structured chat zero shot react-description**

In [20]:
agent_2 = initialize_agent(
    [sum_numbers_from_text], 
    llm, 
    agent="structured-chat-zero-shot-react-description", 
    verbose=True, 
    handle_parsing_errors=True)
response = agent_2.invoke({"input": "additionne 10, 20 et 30"})
print(response)



> Entering new AgentExecutor chain...
{
  "action": "sum_numbers_from_text",
  "action_input": "10, 20 et 30"
}

> Finished chain.
{'input': 'additionne 10, 20 et 30', 'output': '{\n  "action": "sum_numbers_from_text",\n  "action_input": "10, 20 et 30"\n}'}


In [21]:
agent_3 = initialize_agent([sum_numbers_with_complex_output], llm, agent="openai-functions", verbose=True, handle_parsing_errors=True)
response = agent_3.invoke({"input": "Add 10, 20 and 30"})
print(response)



> Entering new AgentExecutor chain...

Invoking: `sum_numbers_with_complex_output` with `{'inputs': 'Add 10, 20 and 30'}`


{'result': 60.0}The sum of 10, 20, and 30 is 60.

> Finished chain.
{'input': 'Add 10, 20 and 30', 'output': 'The sum of 10, 20, and 30 is 60.'}


In [22]:
agent_2 = initialize_agent(
    [add_numbers_with_options],
    llm,
    agent="structured-chat-zero-shot-react-description",
    verbose=True
)
response = agent_2.invoke({
    "input": "Add -10, -20, and -30 using absolute values."
})
print(response)



> Entering new AgentExecutor chain...
{
  "action": "add_numbers_with_options",
  "action_input": {
    "numbers": [-10, -20, -30],
    "absolute": true
  }
}

> Finished chain.
{'input': 'Add -10, -20, and -30 using absolute values.', 'output': '{\n  "action": "add_numbers_with_options",\n  "action_input": {\n    "numbers": [-10, -20, -30],\n    "absolute": true\n  }\n}'}


In [23]:
agent_openai = initialize_agent(
    [add_numbers_with_options],
    llm,
    agent="openai-functions",
    verbose=True
)
response = agent_openai.invoke({
    "input": "Add -10, -20, and -30 using absolute values."
})
print(response)



> Entering new AgentExecutor chain...

Invoking: `add_numbers_with_options` with `{'numbers': [-10, -20, -30], 'absolute': True}`


60.0The sum of -10, -20, and -30 using their absolute values is 60.

> Finished chain.
{'input': 'Add -10, -20, and -30 using absolute values.', 'output': 'The sum of -10, -20, and -30 using their absolute values is 60.'}


### **`create_react_agent`**

In [24]:
from langgraph.prebuilt import create_react_agent

In [ ]:
lang_agent = create_react_agent(model=llm, tools=[sum_numbers_from_text])

## Mise ensemble de plusiers tools

In [20]:
@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string, negates the first number, and successively subtracts 
    the remaining numbers in the list.

    This function is designed to handle input in string format, where numbers are separated 
    by spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, 
    and performs a step-by-step subtraction operation starting with the first number negated.

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string may include spaces, commas, or 
      other delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": -130}

    Notes:
    - Non-numeric characters in the input are ignored.
    - If the input string contains only one valid number, the result will be that number negated.
    - Handles a variety of delimiters (e.g., spaces, commas) but does not validate input formats 
      beyond extracting numeric values.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number negated
    result = -1 * numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [21]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

In [22]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number 
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}

In [23]:
# Testing multiply_tool
multiply_test_input = "2, 3, and four "
multiply_result = multiply_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {multiply_test_input}")
print(f"Output: {multiply_result}")

[2, 3]
2
3
--- Testing MultiplyTool ---
Input: 2, 3, and four 
Output: {'result': 6}


In [24]:
tools = [add_numbers,subtract_numbers, multiply_numbers, divide_numbers]
tools

[StructuredTool(name='add_numbers', description='Adds a list of numbers provided in the input string.\nParameters:\n- inputs (str): \nstring, it should contain numbers that can be extracted and summed.\nReturns:\n- dict: A dictionary with a single key "result" containing the sum of the numbers.\nExample Input:\n"Add the numbers 10, 20, and 30."\nExample Output:\n{"result": 60}', args_schema=<class 'langchain_core.utils.pydantic.add_numbers'>, func=<function add_numbers at 0x000002ADBACBF060>),
 StructuredTool(name='subtract_numbers', description='Extracts numbers from a string, negates the first number, and successively subtracts \nthe remaining numbers in the list.\n\nThis function is designed to handle input in string format, where numbers are separated \nby spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, \nand performs a step-by-step subtraction operation starting with the first number negated.\n\nParameters:\n- inputs (str): \n  A string co

In [25]:
from langgraph.prebuilt import create_react_agent

In [26]:
# !pip install langchain-openai==0.3.16

In [29]:
from langchain_openai import ChatOpenAI

llm_ai = ChatOpenAI(model="gpt-4", api_key=os.getenv("OPENAI_API_KEY"), temperature=0)

In [33]:
# Create the agent with all tools
math_agent = create_react_agent(
    model=llm_ai,
    tools=tools,
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)

In [34]:
response = math_agent.invoke({
    "messages": [("human", "What is 25 divided by 4?")]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

The result of dividing 25 by 4 is 6.25.


In [35]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)

{"result": -130}


In [ ]:
@tool
def new_subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [37]:
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]
# Create the agent with all tools
math_agent_new = create_react_agent(
    model=llm_ai,
    tools=tools_updated,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent",math_agent_new)

agent <langgraph.graph.state.CompiledStateGraph object at 0x000002ADB962B650>


In [38]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

In [39]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = math_agent_new.invoke({"messages": [("human", query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers', 'new_subtract_numbers', 'multiply_numbers', 'divide_numbers']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)


--- Test Case 1: Testing subtraction tool with sequential subtraction. ---
Query: Subtract 100, 20, and 10.
Tool Result: 70
Expected Result: 70
✅ Test Passed: Testing subtraction tool with sequential subtraction.

--- Test Case 2: Testing multiplication tool for a list of numbers. ---
Query: Multiply 2, 3, and 4.
[2, 3, 4]
2
3
4
Tool Result: 24
Expected Result: 24
✅ Test Passed: Testing multiplication tool for a list of numbers.

--- Test Case 3: Testing division tool with sequential division. ---
Query: Divide 100 by 5 and then by 2.
Tool Result: 10.0
Expected Result: 10.0
✅ Test Passed: Testing division tool with sequential division.

--- Test Case 4: Testing subtraction tool with negative results. ---
Query: Subtract 50 from 20.
Tool Result: -30
Expected Result: -30
✅ Test Passed: Testing subtraction tool with negative results.

Correctly passed tests: ['Testing subtraction tool with sequential subtraction.', 'Testing multiplication tool for a list of numbers.', 'Testing division t

In [40]:
from langchain_community.utilities import WikipediaAPIWrapper

In [41]:
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

In [43]:
search_wikipedia.invoke("Who is MBIA NDI Marie Thérèse ?")

"Page: List of Cameroonian writers\nSummary: This is a list of Cameroonian writers.\n\nBoé A-Amang (1938– ), playwright and theatre director[Jahn]\nSeverin Cecile Abega (1955–2008), French-language fiction writer and anthropologist, author of Les Bimanes, Le Bourreau and Entre Terre et Ciel[Gikandi]\nImbolo Mbue (1981– ) novelist\nMarie-Therese Assiga Ahanda, chemist and novelist\nPaul-Charles Atangana (1930– ), French-language poet\nPhilomène Bassek (1957– ), French-language novelist, author of La Tache de Sang[Gikandi]\nFrancis Bebey (1929–2001), author of Les Trois Petits Cireurs, Agatha Moudio'son, The Ashanti Doll, Enfant Pluie and Ministre et le Griot[Gikandi] [Jahn]\nJacques Bengono (1938– ), poet and short story writer[Jahn]\nBate Besong (1954–2007), poet[Gikandi]\nMongo Beti, pseudonym of Alexandre Biyidi Awala (1932–2001), novelist writing in French[Gikandi] [Jahn] [Killam & Rowe]\nCalixthe Beyala (1961– ), novelist writing in French[Gikandi] [Killam & Rowe]\nJacques Bonjawo 

In [44]:
# Update your tools list to include the Wikipedia tool
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers, search_wikipedia]

# Create the agent with all tools including Wikipedia
math_agent_updated = create_react_agent(
    model=llm_ai,
    tools=tools_updated,
    prompt="You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

In [48]:
query = "Quel était la population de la France en 2020? Multiplie cela par 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")

[69081996]
69081996

Message sequence:

--- Message 1 ---
Type: HumanMessage
Content: Quel était la population de la France en 2020? Multiplie cela par 0.75
Name: None

--- Message 2 ---
Type: AIMessage
Content: 
Name: None
Tool calls: [{'name': 'search_wikipedia', 'args': {'query': 'Population of France in 2020'}, 'id': 'call_8H28gNP7AhJMfGWpke12SxyK', 'type': 'tool_call'}]

--- Message 3 ---
Type: ToolMessage
Content: Page: Demographics of France
Summary: The demography of France is monitored by the Institut national d'études démographiques (INED) and the Institut national de la statistique et des études économiques (INSEE). As of 1 January 2026,  in Metropolitan France lived 66,792,845 people, while 2,289,151 lived in overseas France, for a total of 69,081,996 inhabitants in the French Republic. In the 2010s and until 2017, the population of France grew by 1 million people every three years - an average annual increase of 340,000 people, or +0.6%.
France was historically Europe's mo

In [74]:
prompt= "peux-tu me résumer les commentaires de cette vidéo ? https://www.youtube.com/shorts/J5ew4SvjNkM"

In [50]:
llm_ai.invoke(prompt)

AIMessage(content="Je suis désolé, mais en tant qu'IA, je ne peux pas regarder ou analyser le contenu des vidéos. Je peux seulement traiter et comprendre le texte.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 37, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'id': 'chatcmpl-D9sb4BzGHCitEHYa4xN81aJFIS7I4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019c668f-4ce7-77d1-8d4d-7b0356535e71-0', usage_metadata={'input_tokens': 37, 'output_tokens': 38, 'total_tokens': 75, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [51]:
from langchain_community.tools import YouTubeSearchTool

In [53]:
@tool
def search_on_youtube(query: str) -> str:
    """Helps search videos on youtube

    Args:
        query (str): The link of youtube video

    Returns:
        str: The video comments summary
    """
    ytb = YouTubeSearchTool()
    return ytb.run(query)

In [71]:
ytb_agent = create_react_agent(
    model= llm_ai,
    tools=[search_on_youtube],
    prompt= "You are an helpfull assitant which resume comments in youtube video and output the summary."
)

In [75]:
reponse = ytb_agent.invoke({"messages": [("human", prompt)]})

In [76]:
print(reponse["messages"][3].content)

Les commentaires de cette vidéo sont majoritairement positifs. Les spectateurs ont apprécié le contenu et ont exprimé leur admiration pour le créateur de la vidéo. Certains ont également partagé leurs propres expériences et opinions sur le sujet de la vidéo.
